In [ ]:
# ==============================
# 1. Install Libraries
# ==============================

!pip install ultralytics # recomendado para funcionar

In [ ]:
# ==============================
# 2. Import Libraries
# ==============================

from ultralytics import YOLO

import os
import shutil
import random
from pathlib import Path
import yaml

In [ ]:
# ==============================
# 3. Dataset Paths
# ==============================

# Real dataset (your current dataset)
REAL_IMAGES = "CroppedDataset/DatasetReal"
REAL_LABELS = "CroppedDataset/labelsReal"

# Synthetic dataset (future Unreal dataset)
SYN_IMAGES = "" # adicionar quando já tiver o dataset sintético
SYN_LABELS = ""

In [ ]:
# ==============================
# 4. Create YOLO Dataset Split
# ==============================

def split_dataset(images_path, labels_path, output_path):

    images = list(Path(images_path).glob("*.*"))

    random.shuffle(images)

    train_size = int(len(images)*0.7)
    val_size = int(len(images)*0.2)

    datasets = {
        "train": images[:train_size],
        "val": images[train_size:train_size+val_size],
        "test": images[train_size+val_size:]
    }


    for split in datasets:

        os.makedirs(
            f"{output_path}/images/{split}",
            exist_ok=True
        )

        os.makedirs(
            f"{output_path}/labels/{split}",
            exist_ok=True
        )


        for img in datasets[split]:

            label = Path(labels_path) / (img.stem+".txt")


            shutil.copy(
                img,
                f"{output_path}/images/{split}/{img.name}"
            )


            if label.exists():

                shutil.copy(
                    label,
                    f"{output_path}/labels/{split}/{label.name}"
                )

In [ ]:
# ==============================
# 5. Prepare Real Dataset
# ==============================

split_dataset(
    REAL_IMAGES,
    REAL_LABELS,
    "YOLO_REAL"
)

In [ ]:
# ==============================
# 6. Prepare Synthetic Dataset
# ==============================

# Run after Unreal dataset exists

split_dataset(
    SYN_IMAGES,
    SYN_LABELS,
    "YOLO_SYNTHETIC"
)

In [ ]:
# ==============================
# 7. Create YAML File
# ==============================

classes = ["object"]


def create_yaml(path, dataset_name):

    data = {

        "path": os.path.abspath(dataset_name),

        "train":"images/train",
        "val":"images/val",
        "test":"images/test",

        "names":{
            0:"object"
        }

    }


    with open(path,"w") as file:

        yaml.dump(data,file)



create_yaml(
    "real.yaml",
    "YOLO_REAL"
)


create_yaml(
    "synthetic.yaml",
    "YOLO_SYNTHETIC"
)

In [ ]:
# ==============================
# 8. Train YOLO Real Dataset
# ==============================

model_real = YOLO("yolov8n.pt")


model_real.train(

    data="real.yaml",

    epochs=50,

    imgsz=640

)

In [ ]:
# ==============================
# 9. Evaluate Real Dataset
# ==============================

real_results = model_real.val()

real_results

In [ ]:
# ==============================
# 10. Train YOLO Synthetic Dataset
# ==============================

model_syn = YOLO("yolov8n.pt")


model_syn.train(

    data="synthetic.yaml",

    epochs=50,

    imgsz=640

)

In [ ]:
# ==============================
# 11. Test Synthetic Model on Real Images
# ==============================

cross_results = model_syn.val(

    data="real.yaml"

)


cross_results

In [ ]:
# ==============================
# 12. Prediction Example
# ==============================

model_syn.predict(

    source="YOLO_REAL/images/test",

    save=True,

    conf=0.25

)

In [ ]:
# ==============================
# 13. Metrics Comparison
# ==============================

results = {

    "Real trained model":
        model_real.metrics,

    "Synthetic trained model":
        model_syn.metrics

}


results